# Розділ 7: Створення чат-застосунків
## Швидкий старт Microsoft Foundry Models API

Цей блокнот адаптовано з [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), який містить блокноти для доступу до сервісів [Azure OpenAI](aoai-assignment.ipynb).

> **Примітка:** GitHub Models припиняє роботу наприкінці липня 2026 року. Цей блокнот тепер орієнтований на [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), які пропонують той самий каталог моделей для безкоштовного ознайомлення та досвід роботи з Azure AI Inference SDK.


# Огляд  
"Великі мовні моделі — це функції, які відображають текст у текст. Маючи вхідний рядок тексту, велика мовна модель намагається передбачити, який текст з’явиться наступним"(1). Цей блокнот "швидкого старту" ознайомить користувачів з основними концепціями ВММ, основними вимогами до пакету для початку роботи з AML, легким вступом до проєктування підказок, а також кількома короткими прикладами різних випадків використання. 


## Зміст  

[Огляд](#overview)  
[Як використовувати сервіс OpenAI](#how-to-use-openai-service)  
[1. Створення вашого сервісу OpenAI](#1.-creating-your-openai-service)  
[2. Встановлення](#2.-installation)    
[3. Облікові дані](#3.-credentials)  

[Випадки використання](#use-cases)    
[1. Підсумувати текст](#1.-summarize-text)  
[2. Класифікувати текст](#2.-classify-text)  
[3. Генерувати нові назви продуктів](#3.-generate-new-product-names)  
[4. Тонке налаштування класифікатора](#4.fine-tune-a-classifier)  

[Посилання](#references)


### Створіть свій перший запит  
Це коротке завдання забезпечить базове ознайомлення з відправленням запитів до моделі в Microsoft Foundry Models для простої задачі "підсумовування".  


**Кроки**:  
1. Встановіть бібліотеку `azure-ai-inference` у вашому середовищі Python, якщо ви ще цього не зробили.  
2. Завантажте стандартні допоміжні бібліотеки та налаштуйте облікові дані Microsoft Foundry Models.  
3. Виберіть модель для вашого завдання  
4. Створіть простий запит для моделі  
5. Відправте ваш запит до API моделі!  


### 1. Встановлення `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Імпортувати допоміжні бібліотеки та створити облікові дані


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Пошук правильної моделі  
Моделі, такі як GPT-4o та GPT-4o mini, можуть розуміти та генерувати природну мову, і доступні в каталозі моделей Microsoft Foundry разом з моделями від Meta, Mistral, Cohere та Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Проєктування підказок  

"Магія великих мовних моделей полягає в тому, що, навчаючись мінімізувати цю помилку прогнозування на величезній кількості тексту, моделі в результаті навчаються концепціям, корисним для цих прогнозів. Наприклад, вони вивчають концепції, такі як"(1):

* як правильно писати
* як працює граматика
* як перефразовувати
* як відповідати на запитання
* як вести розмову
* як писати багатьма мовами
* як програмувати
* тощо.

#### Як керувати великою мовною моделлю  
"З усіх вхідних даних до великої мовної моделі найбільший вплив має текстова підказка(1).

Великі мовні моделі можна стимулювати для створення вихідних даних кількома способами:

Інструкція: скажіть моделі, чого ви хочете
Завершення: спонукайте модель завершити початок того, що ви хочете
Демонстрація: покажіть моделі, чого ви хочете, за допомогою або:
Кількох прикладів у підказці
Сотень або тисяч прикладів у наборі даних для донавчання"



#### Існує три основні рекомендації щодо створення підказок:

**Показуйте і розповідайте**. Чітко заявляйте, чого ви хочете, через інструкції, приклади або їх комбінацію. Якщо ви хочете, щоб модель впорядкувала список предметів за алфавітом або класифікувала абзац за настроєм, покажіть їй, що саме ви хочете.

**Надавайте якісні дані**. Якщо ви намагаєтеся створити класифікатор або змусити модель дотримуватися певного шаблону, переконайтеся, що прикладів достатньо. Обов’язково перевіряйте свої приклади — модель зазвичай достатньо розумна, щоб розпізнати базові орфографічні помилки і дати відповідь, але вона також може припустити, що це навмисно, і це може вплинути на відповідь.

**Перевірте свої налаштування.** Налаштування temperature та top_p контролюють, наскільки детермінованою буде модель у створенні відповіді. Якщо ви запитуєте відповідь, де є лише одна правильна, ці налаштування варто встановити нижчими. Якщо ви шукаєте більш різноманітні відповіді, можна встановити їх вищими. Найпоширеніша помилка - це припускати, що ці налаштування відповідають за "кмітливість" чи "творчість".


Джерело: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Надіслати!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Повторити той самий виклик, як порівнюються результати? 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Підсумувати текст  
#### Виклик  
Підсумуйте текст, додавши «tl;dr:» у кінець текстового уривка. Зверніть увагу, як модель розуміє, як виконувати низку завдань без додаткових інструкцій. Ви можете експериментувати з більш описовими підказками, ніж tl;dr, щоб змінити поведінку моделі та налаштувати отримане підсумування(3).  

Останні дослідження продемонстрували значний прорив у багатьох NLP-завданнях і бенчмарках шляхом попереднього навчання на великому корпусі тексту, а потім тонкого налаштування на конкретне завдання. Хоча зазвичай архітектура не залежить від завдання, цей метод все ще вимагає наборів даних для тонкого налаштування, характерних для конкретного завдання, що містять тисячі або десятки тисяч прикладів. Натомість люди здатні загалом виконувати нове мовне завдання, маючи лише кілька прикладів або прості інструкції — те, з чим нинішні системи NLP все ще значною мірою борються. Тут ми демонструємо, що масштабування мовних моделей значно покращує універсальну, малооб’ємну продуктивність, інколи навіть досягаючи конкуренції з попередніми найкращими методами тонкого налаштування.  



Підсумок  


# Вправи для кількох випадків використання  
1. Підсумувати текст  
2. Класифікувати текст  
3. Створити нові назви продуктів


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Класифікувати текст  
#### Завдання  
Класифікуйте елементи за категоріями, що надаються під час інференсу. У наведеному нижче прикладі ми надаємо як категорії, так і текст для класифікації у підказці(*playground_reference). 

Запит клієнта: Привіт, недавно зламалась одна з клавіш на клавіатурі мого ноутбука, і мені потрібна заміна:

Класифікована категорія:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Створення нових назв продуктів
#### Виклик
Створіть назви продуктів на основі прикладів слів. Тут ми включаємо в підказку інформацію про продукт, для якого ми збираємося генерувати назви. Також ми надаємо схожий приклад, щоб показати зразок, який ми хочемо отримати. Ми також встановили високе значення температури, щоб збільшити випадковість і отримати більш інноваційні відповіді.

Опис продукту: Домашній прилад для приготування молочних коктейлів
Стартові слова: швидкий, здоровий, компактний.
Назви продуктів: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Опис продукту: Пара взуття, що підходить для будь-якого розміру ступні.
Стартові слова: адаптивний, підходить, омні-фіт.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Посилання  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Портал Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Найкращі практики для тонкої настройки моделей GPT для класифікації тексту](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Для додаткової допомоги  
[Команда комерціалізації OpenAI](AzureOpenAITeam@microsoft.com) 


# Учасники
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
